# Decision Tree: Adult Income classification

This notebook uses the Adult Income dataset from the `models.md` table to predict whether income exceeds $50K.
It practices encoding mixed categorical/numerical features and explores entropy/Gini splitting with pruning.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# ---------- Load ----------
cols = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
        'income']
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, header=None, names=cols, skipinitialspace=True)

# ---------- Clean ----------
# Remove rows with missing values marked as '?'.
df = df.replace('?', pd.NA).dropna()
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

# ---------- Split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- Encode ----------
# Fit encoding on training data only to avoid leakage.
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# ---------- Decision Tree ----------
model = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
model.fit(X_train_enc, y_train)
y_pred = model.predict(X_test_enc)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred):.4f}')
print('\nClassification report:\n',
      classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['<=50K', '>50K']).plot(cmap='Blues')
plt.title('Decision Tree (entropy, max_depth=5): confusion matrix')
plt.show()

## Decision Tree: splitting criteria, pruning, and evaluation

### Notation

- $S$: set of samples at a node.
- $c \in \{0, 1\}$: class ($0$ = ≤50K, $1$ = >50K).
- $p_c = |S_c| / |S|$: proportion of class $c$ in $S$.
- $A$: a candidate feature to split on; $S_v$: subset where feature $A$ takes value $v$.

### Entropy

Entropy measures impurity — the uncertainty about the class label:

$$H(S) = -\sum_{c} p_c \log_2 p_c$$

$H = 0$ when all samples belong to one class (pure). $H = 1$ for a binary split with equal proportions.

### Gini impurity

An alternative impurity measure:

$$G(S) = 1 - \sum_{c} p_c^2$$

Gini and entropy usually produce similar trees. Gini is slightly faster (no logarithm) and is scikit-learn's default.

### Information Gain

The tree selects the split that maximizes information gain — the reduction in impurity:

$$\mathrm{IG}(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|}\, H(S_v)$$

**Maximize** IG: a higher value means the split produces purer child nodes.

### Gain Ratio

IG is biased toward features with many distinct values. Gain ratio corrects for this:

$$\mathrm{GainRatio}(S, A) = \frac{\mathrm{IG}(S, A)}{\mathrm{SplitInfo}(S, A)}, \qquad \mathrm{SplitInfo} = -\sum_{v} \frac{|S_v|}{|S|}\log_2 \frac{|S_v|}{|S|}$$

### Pruning

Unpruned trees tend to overfit. Two strategies control tree complexity:

**Pre-pruning** (stop growing early):
- `max_depth`: maximum tree depth.
- `min_samples_split`: minimum samples to attempt a split.
- `min_samples_leaf`: minimum samples in any leaf.

**Post-pruning** (grow full, then prune):
- `ccp_alpha` (cost-complexity pruning): higher $\alpha$ removes more nodes; select via cross-validation.

### Key hyperparameters

| Parameter | Typical values | Effect |
| --- | --- | --- |
| `criterion` | `'gini'`, `'entropy'`, `'log_loss'` | Impurity function used for splits |
| `max_depth` | 3–20 or `None` | Limits tree depth; primary pre-pruning control |
| `min_samples_split` | 2–20 | Minimum samples required to split an internal node |
| `min_samples_leaf` | 1–10 | Minimum samples in each leaf |
| `ccp_alpha` | 0.0–0.05 | Cost-complexity pruning parameter; higher = simpler tree |

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** Accuracy, Precision, Recall, and $F_1$: each ranges from $0$ (worst) to $1$ (best). For income prediction, >50K recall measures how many high earners the model catches, and >50K precision measures how often a predicted high earner truly earns >50K.